# NB8 — Efficiency & Training-Cost Table

**GastroNet project — Part D, C.2 item 4.** Since NB6 already showed none of the 4 architectures are significantly more *accurate*, this notebook answers the follow-up question a reviewer would actually ask: **if they're tied on accuracy, which one would you deploy?** Params, FLOPs, epochs-to-converge, and wall-clock training time — combined with NB6's accuracy and NB7's ECE (if you've run it) into one deployment-recommendation table.

### What this needed that wasn't available before
Wall-clock time comes from `experiments_manifest.json`, read via your actual `checkpoint_utils.py` (now uploaded — this section is built against its real schema, not guessed). A training run can be interrupted and resumed multiple times, so this notebook doesn't just diff first/last timestamp — it sums the active duration of every `started`/`resumed` → next-event segment and *skips* the dead time between an `interrupted` event and the next `resumed` one, so a run that sat disconnected overnight doesn't get counted as 8 hours of training.

### Caveat, stated up front
FLOP counting for `nn.MultiheadAttention` is approximate in most tools (`fvcore` included) — treat the FLOPs column as directionally useful for comparing architectures, not as an exact number to quote without caveat in the paper. Params and wall-clock time don't have this issue.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install timm fvcore --break-system-packages -q

import os, sys, json, glob, re
from datetime import datetime
from collections import defaultdict
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision
import timm

EXPERIMENTS_ROOT = "/content/drive/MyDrive/gastronet_experiments"
MODEL_FAMILIES = ["cnn_only_v2", "vit_only_v2", "hybrid_concat_v2", "hybrid_crossattn_v2"]
SEEDS = [42, 123, 7]
CLASS_NAMES = ["Diverticulosis", "Neoplasm", "Peritonitis", "Ureters"]
IMG_SIZE_CNN = 448
IMG_SIZE_VIT = 224
NUM_EPOCHS_CEILING = 30   # matches NUM_EPOCHS in NB1/NB2/NB3/NB5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

sys.path.insert(0, EXPERIMENTS_ROOT)
import checkpoint_utils as cku
print("checkpoint_utils imported OK")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.3 MB/s eta 0:00:00
Device: cpu
checkpoint_utils imported OK


## 2. Model class definitions (verbatim, same as NB7)

Only what's needed for architecture-level params/FLOPs — no checkpoint loading in this notebook, so these are built fresh (pretrained weights are loaded because the builders hardcode `pretrained=True`, but the actual weight *values* don't affect a param or FLOP count either way).

In [3]:
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

def build_cnn_model(num_classes=len(CLASS_NAMES)):
    weights = EfficientNet_B4_Weights.IMAGENET1K_V1
    model = efficientnet_b4(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def build_vit_model(num_classes=len(CLASS_NAMES)):
    return timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=num_classes)

In [4]:
class HybridConcatModel(nn.Module):
    def __init__(self, num_classes=len(CLASS_NAMES)):
        super().__init__()
        cnn = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.cnn_features = cnn.features
        self.cnn_pool = cnn.avgpool
        cnn_feature_dim = cnn.classifier[1].in_features
        self.vit = timm.create_model("vit_small_patch16_224", pretrained=True, num_classes=0)
        vit_feature_dim = self.vit.num_features
        fused_dim = cnn_feature_dim + vit_feature_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, num_classes),
        )

    def forward(self, cnn_img, vit_img):
        cnn_feat = torch.flatten(self.cnn_pool(self.cnn_features(cnn_img)), 1)
        vit_feat = self.vit(vit_img)
        return self.classifier(torch.cat([cnn_feat, vit_feat], dim=1))

def build_hybrid_concat_model(num_classes=len(CLASS_NAMES)):
    return HybridConcatModel(num_classes)

In [5]:
class CNNSpatialEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        backbone = torchvision.models.efficientnet_b4(weights=weights)
        self.features = backbone.features
        self.out_channels = 1792

    def forward(self, x):
        return self.features(x)


class ViTTokenEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.vit = timm.create_model("vit_small_patch16_224", pretrained=pretrained, num_classes=0)
        self.embed_dim = self.vit.embed_dim
        self.num_prefix_tokens = getattr(self.vit, "num_prefix_tokens", 1)

    def forward(self, x):
        tokens = self.vit.forward_features(x)
        return tokens[:, self.num_prefix_tokens:, :]


class Learned2DPositionalEmbedding(nn.Module):
    def __init__(self, grid_h, grid_w, dim):
        super().__init__()
        self.grid_h, self.grid_w = grid_h, grid_w
        self.row_embed = nn.Parameter(torch.zeros(grid_h, dim))
        self.col_embed = nn.Parameter(torch.zeros(grid_w, dim))
        nn.init.trunc_normal_(self.row_embed, std=0.02)
        nn.init.trunc_normal_(self.col_embed, std=0.02)

    def forward(self):
        pos = self.row_embed[:, None, :] + self.col_embed[None, :, :]
        return pos.reshape(self.grid_h * self.grid_w, -1)


class CrossAttentionBlock(nn.Module):
    def __init__(self, d_model=256, num_heads=4, ff_mult=4, dropout=0.3):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model * ff_mult, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, cnn_tokens, vit_tokens):
        attn_out, attn_weights = self.attn(query=cnn_tokens, key=vit_tokens, value=vit_tokens, need_weights=True)
        x = self.norm1(cnn_tokens + self.dropout(attn_out))
        x = self.norm2(x + self.ff(x))
        return x, attn_weights


class HybridCrossAttnModel(nn.Module):
    def __init__(self, num_classes=4, d_model=256, num_heads=4, num_layers=2,
                 cnn_grid=None, dropout=0.3, pretrained_backbones=True):
        super().__init__()
        self.cnn_encoder = CNNSpatialEncoder(pretrained=pretrained_backbones)
        self.vit_encoder = ViTTokenEncoder(pretrained=pretrained_backbones)
        cnn_channels = self.cnn_encoder.out_channels
        vit_dim = self.vit_encoder.embed_dim
        grid_h, grid_w = cnn_grid
        self.cnn_proj = nn.Linear(cnn_channels, d_model)
        self.pos_embed = Learned2DPositionalEmbedding(grid_h, grid_w, d_model)
        self.vit_proj = nn.Linear(vit_dim, d_model)
        self.blocks = nn.ModuleList([
            CrossAttentionBlock(d_model=d_model, num_heads=num_heads, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, num_classes))
        self.grid_h, self.grid_w = grid_h, grid_w

    def forward(self, cnn_input, vit_input):
        cnn_feat_map = self.cnn_encoder(cnn_input)
        B, C, H, W = cnn_feat_map.shape
        cnn_tokens = self.cnn_proj(cnn_feat_map.flatten(2).transpose(1, 2))
        cnn_tokens = cnn_tokens + self.pos_embed()[None, :, :]
        vit_tokens = self.vit_proj(self.vit_encoder(vit_input))
        x = cnn_tokens
        for block in self.blocks:
            x, _ = block(x, vit_tokens)
        return self.classifier(x.mean(dim=1))


def _probe_cnn_grid():
    probe = CNNSpatialEncoder(pretrained=False).to(device).eval()
    with torch.no_grad():
        out = probe(torch.randn(2, 3, IMG_SIZE_CNN, IMG_SIZE_CNN, device=device))
    grid = (out.shape[-2], out.shape[-1])
    del probe, out
    return grid

def build_crossattn_model(cnn_grid=None):
    return HybridCrossAttnModel(
        num_classes=len(CLASS_NAMES), d_model=256, num_heads=4, num_layers=2,
        cnn_grid=cnn_grid or _probe_cnn_grid(), dropout=0.3, pretrained_backbones=True,
    )

print("Model classes ready.")

Model classes ready.


## 3. Parameter counts

Total and trainable params per architecture, plus a component breakdown (backbone(s) vs. fusion/head) — useful since the fusion blocks themselves are tiny relative to the two pretrained backbones they sit on top of.

In [6]:
def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

params_rows = []

m = build_cnn_model()
total, trainable = count_params(m)
backbone_total, _ = count_params(m.features)
head_total, _ = count_params(m.classifier)
params_rows.append({"model_family": "cnn_only_v2", "total_params": total, "trainable_params": trainable,
                     "backbone_params": backbone_total, "head_params": head_total, "fusion_params": 0})
del m

m = build_vit_model()
total, trainable = count_params(m)
head_total, _ = count_params(m.head)
backbone_total = total - head_total
params_rows.append({"model_family": "vit_only_v2", "total_params": total, "trainable_params": trainable,
                     "backbone_params": backbone_total, "head_params": head_total, "fusion_params": 0})
del m

m = build_hybrid_concat_model()
total, trainable = count_params(m)
cnn_backbone, _ = count_params(m.cnn_features)
vit_backbone, _ = count_params(m.vit)
head_total, _ = count_params(m.classifier)
params_rows.append({"model_family": "hybrid_concat_v2", "total_params": total, "trainable_params": trainable,
                     "backbone_params": cnn_backbone + vit_backbone, "head_params": head_total, "fusion_params": 0})
del m

m = build_crossattn_model()
total, trainable = count_params(m)
cnn_backbone, _ = count_params(m.cnn_encoder)
vit_backbone, _ = count_params(m.vit_encoder)
fusion_total = (count_params(m.cnn_proj)[0] + count_params(m.pos_embed)[0] +
                count_params(m.vit_proj)[0] + count_params(m.blocks)[0])
head_total, _ = count_params(m.classifier)
params_rows.append({"model_family": "hybrid_crossattn_v2", "total_params": total, "trainable_params": trainable,
                     "backbone_params": cnn_backbone + vit_backbone, "head_params": head_total, "fusion_params": fusion_total})
del m

params_df = pd.DataFrame(params_rows)
params_df_display = params_df.copy()
for c in ["total_params", "trainable_params", "backbone_params", "head_params", "fusion_params"]:
    params_df_display[c] = (params_df_display[c] / 1e6).round(2)
params_df_display = params_df_display.rename(columns={c: c + "_M" for c in
    ["total_params", "trainable_params", "backbone_params", "head_params", "fusion_params"]})
print("Parameter counts (millions):")
print(params_df_display.to_string(index=False))

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 156MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Parameter counts (millions):
       model_family  total_params_M  trainable_params_M  backbone_params_M  head_params_M  fusion_params_M
        cnn_only_v2           17.56               17.56              17.55           0.01             0.00
        vit_only_v2           21.67               21.67              21.67           0.00             0.00
   hybrid_concat_v2           39.77               39.77              39.21           0.56             0.00
hybrid_crossattn_v2           41.36               41.36              39.21           0.00             2.14


## 4. FLOPs

One forward pass per architecture at its real input size(s) (batch=1). `fvcore` prints "Unsupported operator" warnings for ops it can't count exactly (this will include `nn.MultiheadAttention`'s internals for `hybrid_crossattn_v2`) — those warnings are captured and surfaced in the table below as `uncounted_ops`, rather than silently hidden, so you know exactly how much to trust each number.

In [7]:
from fvcore.nn import FlopCountAnalysis

def count_flops(model, *inputs):
    """Wrapped defensively -- fvcore's exact API (esp. unsupported_ops_warnings/
    unsupported_ops) has shifted slightly across versions, and this notebook was
    written without being able to test against your installed version directly.
    A method-name mismatch here degrades to 'uncounted ops unknown' rather than
    crashing the whole cell."""
    model.eval()
    with torch.no_grad():
        flop_analysis = FlopCountAnalysis(model, tuple(inputs))
        try:
            flop_analysis.unsupported_ops_warnings(False)
        except AttributeError:
            pass
        total_flops = flop_analysis.total()
        try:
            uncounted = flop_analysis.unsupported_ops()
        except AttributeError:
            uncounted = {}
    return total_flops, uncounted

flop_rows = []
dummy_cnn = torch.randn(1, 3, IMG_SIZE_CNN, IMG_SIZE_CNN)
dummy_vit = torch.randn(1, 3, IMG_SIZE_VIT, IMG_SIZE_VIT)

for fam in MODEL_FAMILIES:
    try:
        if fam == "cnn_only_v2":
            model = build_cnn_model()
            flops, uncounted = count_flops(model, dummy_cnn)
        elif fam == "vit_only_v2":
            model = build_vit_model()
            flops, uncounted = count_flops(model, dummy_vit)
        elif fam == "hybrid_concat_v2":
            model = build_hybrid_concat_model()
            flops, uncounted = count_flops(model, dummy_cnn, dummy_vit)
        else:
            model = build_crossattn_model()
            flops, uncounted = count_flops(model, dummy_cnn, dummy_vit)
        n_uncounted_types = len(uncounted)
        flop_rows.append({"model_family": fam, "gflops": flops / 1e9, "uncounted_op_types": n_uncounted_types,
                           "uncounted_ops_detail": dict(uncounted)})
        del model
    except Exception as e:
        print(f"[WARNING] FLOP counting failed for {fam}: {e}")
        flop_rows.append({"model_family": fam, "gflops": np.nan, "uncounted_op_types": np.nan, "uncounted_ops_detail": {}})

flops_df = pd.DataFrame(flop_rows)
print("GFLOPs per forward pass (batch=1):")
print(flops_df[["model_family", "gflops", "uncounted_op_types"]].to_string(
    index=False, formatters={"gflops": "{:.2f}".format}))
print("\nUncounted op types (non-empty means that model's GFLOPs is an UNDER-estimate):")
for _, row in flops_df.iterrows():
    if row["uncounted_ops_detail"]:
        print(f"  {row['model_family']}: {row['uncounted_ops_detail']}")

features.1.0.stochastic_depth, features.1.1.stochastic_depth, features.2.0.stochastic_depth, features.2.1.stochastic_depth, features.2.2.stochastic_depth, features.2.3.stochastic_depth, features.3.0.stochastic_depth, features.3.1.stochastic_depth, features.3.2.stochastic_depth, features.3.3.stochastic_depth, features.4.0.stochastic_depth, features.4.1.stochastic_depth, features.4.2.stochastic_depth, features.4.3.stochastic_depth, features.4.4.stochastic_depth, features.4.5.stochastic_depth, features.5.0.stochastic_depth, features.5.1.stochastic_depth, features.5.2.stochastic_depth, features.5.3.stochastic_depth, features.5.4.stochastic_depth, features.5.5.stochastic_depth, features.6.0.stochastic_depth, features.6.1.stochastic_depth, features.6.2.stochastic_depth, features.6.3.stochastic_depth, features.6.4.stochastic_depth, features.6.5.stochastic_depth, features.6.6.stochastic_depth, features.6.7.stochastic_depth, features.7.0.stochastic_depth, features.7.1.stochastic_depth
blocks.0.

GFLOPs per forward pass (batch=1):
       model_family gflops  uncounted_op_types
        cnn_only_v2   6.16                   5
        vit_only_v2   4.25                   3
   hybrid_concat_v2  10.41                   7
hybrid_crossattn_v2  10.87                  11

Uncounted op types (non-empty means that model's GFLOPs is an UNDER-estimate):
  cnn_only_v2: {'aten::silu_': 96, 'aten::sigmoid': 32, 'aten::mul': 32, 'aten::add_': 25, 'aten::dropout_': 1}
  vit_only_v2: {'aten::add': 25, 'aten::scaled_dot_product_attention': 12, 'aten::gelu': 12}
  hybrid_concat_v2: {'aten::silu_': 96, 'aten::sigmoid': 32, 'aten::mul': 32, 'aten::add_': 25, 'aten::add': 25, 'aten::scaled_dot_product_attention': 12, 'aten::gelu': 12}
  hybrid_crossattn_v2: {'aten::silu_': 96, 'aten::sigmoid': 32, 'aten::mul': 46, 'aten::add_': 25, 'aten::add': 31, 'aten::scaled_dot_product_attention': 12, 'aten::gelu': 14, 'aten::div': 2, 'aten::unflatten': 2, 'aten::softmax': 2, 'aten::mean': 1}


## 5. Epochs-to-converge

`best_epoch` comes straight from each seed's `results.json` (already saved there). Whether early stopping actually triggered comes from `history.json`'s length — if it's less than the `NUM_EPOCHS=30` ceiling, `PATIENCE=6` stopped it early; if it's exactly 30, it ran to the ceiling without triggering early stopping.

In [8]:
def normalize(name):
    return name.lower().replace("_", "").replace("-", "")

def find_seed_file(results_root, family, seed, filename):
    nested = os.path.join(results_root, family, f"seed_{seed}", filename)
    if os.path.exists(nested):
        return nested
    flat = os.path.join(results_root, f"{family}_seed{seed}_{filename}")
    if os.path.exists(flat):
        return flat
    target = normalize(family)
    for fp in glob.glob(os.path.join(results_root, "**", filename), recursive=True):
        if str(seed) in fp and normalize(fp).find(target[:12]) != -1:
            return fp
    return None

epoch_rows = []
for fam in MODEL_FAMILIES:
    for seed in SEEDS:
        results_fp = find_seed_file(EXPERIMENTS_ROOT, fam, seed, "results.json")
        assert results_fp, f"Could not find results.json for {fam} seed {seed}"
        with open(results_fp) as f:
            results = json.load(f)
        best_epoch = results["best_epoch"]

        history_fp = find_seed_file(EXPERIMENTS_ROOT, fam, seed, "history.json")
        if history_fp:
            with open(history_fp) as f:
                history = json.load(f)
            epochs_run = len(history.get("val_acc", []))
            early_stopped = epochs_run < NUM_EPOCHS_CEILING
        else:
            print(f"  [WARNING] no history.json found for {fam} seed {seed} -- "
                  f"epochs_run/early_stopped will be blank for this row.")
            epochs_run, early_stopped = None, None

        epoch_rows.append({
            "model_family": fam, "seed": seed, "best_epoch": best_epoch,
            "epochs_run": epochs_run, "early_stopped": early_stopped,
        })

epoch_df = pd.DataFrame(epoch_rows)
epoch_summary = epoch_df.groupby("model_family")["best_epoch"].agg(["mean", "std"]).reset_index()
epoch_summary.columns = ["model_family", "best_epoch_mean", "best_epoch_std"]
print(epoch_df.to_string(index=False))
print("\nMean +/- std best_epoch across 3 seeds:")
print(epoch_summary.to_string(index=False, formatters={"best_epoch_mean": "{:.1f}".format, "best_epoch_std": "{:.1f}".format}))

       model_family  seed  best_epoch  epochs_run  early_stopped
        cnn_only_v2    42           3          10           True
        cnn_only_v2   123           3          10           True
        cnn_only_v2     7          10          17           True
        vit_only_v2    42           6          13           True
        vit_only_v2   123           0           7           True
        vit_only_v2     7           7          14           True
   hybrid_concat_v2    42           6          13           True
   hybrid_concat_v2   123           3          10           True
   hybrid_concat_v2     7           5          12           True
hybrid_crossattn_v2    42           2           9           True
hybrid_crossattn_v2   123           3          10           True
hybrid_crossattn_v2     7          11          18           True

Mean +/- std best_epoch across 3 seeds:
       model_family best_epoch_mean best_epoch_std
        cnn_only_v2             5.3            4.0
   hybrid_co

## 6. Wall-clock training time (`experiments_manifest.json`)

Built against your actual `checkpoint_utils.py`: `log_run_event` appends one entry per status transition (`started`/`resumed`/`completed`/`interrupted`) with a `%Y-%m-%d %H:%M:%S` timestamp, `_load_manifest` returns `{"runs": [...]}`. A run can be interrupted and resumed multiple times across sessions/accounts, so total active training time is the **sum of every `started`/`resumed` → next-event gap** — time between an `interrupted` event and the next `resumed` one is skipped (that's disconnected dead time, not training time).

In [9]:
MANIFEST_JSON_PATH = os.path.join(EXPERIMENTS_ROOT, "experiments_manifest.json")
assert os.path.exists(MANIFEST_JSON_PATH), f"No manifest found at {MANIFEST_JSON_PATH}"

with open(MANIFEST_JSON_PATH) as f:
    manifest = json.load(f)
runs = manifest["runs"]
print(f"Loaded {len(runs)} manifest events total.")

TS_FORMAT = "%Y-%m-%d %H:%M:%S"

def active_seconds_for_key(model_family, seed):
    """Sum of active (started/resumed -> next event) durations for one
    (model_family, seed), skipping any interrupted->resumed dead time."""
    events = [r for r in runs if r["model_family"] == model_family and r["seed"] == seed]
    events = sorted(events, key=lambda r: datetime.strptime(r["timestamp"], TS_FORMAT))
    total = 0.0
    for e1, e2 in zip(events[:-1], events[1:]):
        if e1["status"] in ("started", "resumed"):
            t1 = datetime.strptime(e1["timestamp"], TS_FORMAT)
            t2 = datetime.strptime(e2["timestamp"], TS_FORMAT)
            total += (t2 - t1).total_seconds()
    return total, len(events)

time_rows = []
for fam in MODEL_FAMILIES:
    for seed in SEEDS:
        secs, n_events = active_seconds_for_key(fam, seed)
        if n_events == 0:
            print(f"  [WARNING] no manifest events found for {fam} seed {seed} -- "
                  f"wall-clock time will be blank for this row.")
        time_rows.append({
            "model_family": fam, "seed": seed,
            "wall_clock_minutes": secs / 60 if n_events else None,
            "n_manifest_events": n_events,
        })

time_df = pd.DataFrame(time_rows)
time_summary = time_df.groupby("model_family")["wall_clock_minutes"].agg(["mean", "std"]).reset_index()
time_summary.columns = ["model_family", "wall_clock_minutes_mean", "wall_clock_minutes_std"]
print(time_df.to_string(index=False, formatters={"wall_clock_minutes": lambda x: f"{x:.1f}" if pd.notnull(x) else "N/A"}))
print("\nMean +/- std wall-clock minutes across 3 seeds:")
print(time_summary.to_string(index=False, formatters={c: "{:.1f}".format for c in ["wall_clock_minutes_mean","wall_clock_minutes_std"]}))

Loaded 36 manifest events total.
       model_family  seed wall_clock_minutes  n_manifest_events
        cnn_only_v2    42               16.8                  2
        cnn_only_v2   123               15.8                  2
        cnn_only_v2     7               26.7                  2
        vit_only_v2    42               10.0                  2
        vit_only_v2   123                5.3                  2
        vit_only_v2     7               10.8                  2
   hybrid_concat_v2    42               33.5                  2
   hybrid_concat_v2   123               24.6                  2
   hybrid_concat_v2     7               29.6                  2
hybrid_crossattn_v2    42               18.9                  2
hybrid_crossattn_v2   123               21.5                  2
hybrid_crossattn_v2     7               39.6                  2

Mean +/- std wall-clock minutes across 3 seeds:
       model_family wall_clock_minutes_mean wall_clock_minutes_std
        cnn_only_v2

## 7. Combined deployment-recommendation table

One row per architecture: accuracy (from NB6/`results.json`), params, FLOPs, epochs-to-converge, wall-clock time, and calibration (ECE, from NB7's saved output if you've run it — this column is skipped gracefully if not). This is the actual payoff table: since NB6 already showed none of these are significantly more *accurate*, this is what lets you answer the supervisor's B.2 pushback with "here's which one you'd actually deploy" instead of just "the ceiling is the finding."

In [10]:
acc_rows = []
for fam in MODEL_FAMILIES:
    accs = []
    for seed in SEEDS:
        fp = find_seed_file(EXPERIMENTS_ROOT, fam, seed, "results.json")
        with open(fp) as f:
            accs.append(json.load(f)["test_accuracy"])
    acc_rows.append({"model_family": fam, "accuracy_mean": np.mean(accs), "accuracy_std": np.std(accs, ddof=1)})
acc_df = pd.DataFrame(acc_rows)

ece_path = os.path.join(EXPERIMENTS_ROOT, "analysis", "nb7_ece_summary.csv")
if os.path.exists(ece_path):
    ece_df = pd.read_csv(ece_path).rename(columns={"mean": "ece_mean", "std": "ece_std"})
    print("Loaded NB7's ECE summary -- including calibration in the combined table.")
else:
    ece_df = pd.DataFrame({"model_family": MODEL_FAMILIES, "ece_mean": [np.nan]*4, "ece_std": [np.nan]*4})
    print(f"No NB7 ECE output found at {ece_path} -- calibration column will be blank. "
          f"Run NB7 and re-run this cell to fill it in.")

combined = (acc_df
    .merge(params_df[["model_family", "total_params"]], on="model_family")
    .merge(flops_df[["model_family", "gflops"]], on="model_family")
    .merge(epoch_summary, on="model_family")
    .merge(time_summary, on="model_family")
    .merge(ece_df[["model_family", "ece_mean", "ece_std"]], on="model_family"))
combined["total_params_M"] = (combined["total_params"] / 1e6).round(1)
combined = combined.drop(columns=["total_params"])
combined = combined.sort_values("accuracy_mean", ascending=False).reset_index(drop=True)

display_cols = ["model_family", "accuracy_mean", "accuracy_std", "total_params_M", "gflops",
                 "best_epoch_mean", "wall_clock_minutes_mean", "ece_mean"]
print("\n=== Combined deployment table ===")
print(combined[display_cols].to_string(index=False, formatters={
    "accuracy_mean": "{:.4f}".format, "accuracy_std": "{:.4f}".format,
    "total_params_M": "{:.1f}".format, "gflops": "{:.2f}".format,
    "best_epoch_mean": "{:.1f}".format, "wall_clock_minutes_mean": "{:.1f}".format,
    "ece_mean": lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A",
}))

Loaded NB7's ECE summary -- including calibration in the combined table.

=== Combined deployment table ===
       model_family accuracy_mean accuracy_std total_params_M gflops best_epoch_mean wall_clock_minutes_mean ece_mean
        cnn_only_v2        0.9800       0.0025           17.6   6.16             5.3                    19.8   0.0126
   hybrid_concat_v2        0.9775       0.0025           39.8  10.41             4.7                    29.2   0.0183
        vit_only_v2        0.9717       0.0088           21.7   4.25             4.3                     8.7   0.0236
hybrid_crossattn_v2        0.9683       0.0076           41.4  10.87             5.3                    26.7   0.0594


## 8. Auto-generated interpretation

In [11]:
acc_range = combined["accuracy_mean"].max() - combined["accuracy_mean"].min()
cheapest_params = combined.loc[combined["total_params_M"].idxmin()]
fastest_train = combined.loc[combined["wall_clock_minutes_mean"].idxmin()]
lowest_flops = combined.loc[combined["gflops"].idxmin()] if combined["gflops"].notna().any() else None

print(f"Accuracy spread across all 4 architectures: {acc_range:.4f} "
      f"({acc_range*100:.2f} points) -- and per NB6, none of this spread is "
      f"statistically significant.")
print(f"\nSmallest model: {cheapest_params['model_family']} ({cheapest_params['total_params_M']:.1f}M params)")
print(f"Fastest to train: {fastest_train['model_family']} "
      f"({fastest_train['wall_clock_minutes_mean']:.1f} min/seed mean)")
if lowest_flops is not None:
    print(f"Lowest FLOPs: {lowest_flops['model_family']} ({lowest_flops['gflops']:.2f} GFLOPs/forward pass)")

if combined["ece_mean"].notna().any():
    best_cal = combined.loc[combined["ece_mean"].idxmin()]
    print(f"Best-calibrated: {best_cal['model_family']} (ECE={best_cal['ece_mean']:.4f})")

print("\nSince NB6 already ruled out a significant accuracy difference, the "
      "deployment case for any one of these architectures has to rest on this "
      "table -- params/FLOPs/training cost/calibration -- not on accuracy. "
      "If hybrid_crossattn_v2 is both the most expensive to train AND not "
      "significantly more accurate, that itself is a legitimate, citable "
      "conclusion for the paper's deployment recommendation.")

Accuracy spread across all 4 architectures: 0.0117 (1.17 points) -- and per NB6, none of this spread is statistically significant.

Smallest model: cnn_only_v2 (17.6M params)
Fastest to train: vit_only_v2 (8.7 min/seed mean)
Lowest FLOPs: vit_only_v2 (4.25 GFLOPs/forward pass)
Best-calibrated: cnn_only_v2 (ECE=0.0126)

Since NB6 already ruled out a significant accuracy difference, the deployment case for any one of these architectures has to rest on this table -- params/FLOPs/training cost/calibration -- not on accuracy. If hybrid_crossattn_v2 is both the most expensive to train AND not significantly more accurate, that itself is a legitimate, citable conclusion for the paper's deployment recommendation.


## 9. Save outputs

In [12]:
analysis_dir = os.path.join(EXPERIMENTS_ROOT, "analysis")
os.makedirs(analysis_dir, exist_ok=True)

params_df.to_csv(os.path.join(analysis_dir, "nb8_params.csv"), index=False)
flops_df[["model_family", "gflops", "uncounted_op_types"]].to_csv(os.path.join(analysis_dir, "nb8_flops.csv"), index=False)
epoch_df.to_csv(os.path.join(analysis_dir, "nb8_epochs_per_seed.csv"), index=False)
time_df.to_csv(os.path.join(analysis_dir, "nb8_wallclock_per_seed.csv"), index=False)
combined.to_csv(os.path.join(analysis_dir, "nb8_combined_deployment_table.csv"), index=False)

print(f"Saved to {analysis_dir}:")
print("  nb8_params.csv")
print("  nb8_flops.csv")
print("  nb8_epochs_per_seed.csv")
print("  nb8_wallclock_per_seed.csv")
print("  nb8_combined_deployment_table.csv   <- the main deliverable")

Saved to /content/drive/MyDrive/gastronet_experiments/analysis:
  nb8_params.csv
  nb8_flops.csv
  nb8_epochs_per_seed.csv
  nb8_wallclock_per_seed.csv
  nb8_combined_deployment_table.csv   <- the main deliverable


## Next steps

1. **If NB7 hasn't been run yet**, re-run Section 7's merge cell afterward to pull calibration into the combined table.
2. **C.2 item 5** (own-model ensembling) is the only remaining optional item from Part D — only worth doing if time allows.
3. **Read the DeepGI paper** (B.3) before finalizing how novelty is worded in the writeup.
4. The combined table here plus NB6's significance results plus NB7's qualitative findings (t-SNE separation, calibration, the unanimous-error images) is the actual content of the paper's results section at this point.